# Notebook 04 — Validation benchmark: random, grouped, and leave-out splits

**Project:** CMT Path A — leakage-audited multi-ion computed insertion-electrode ML benchmark
**Notebook:** `04_validation_benchmark_random_grouped_leaveout.ipynb`

This resource-aware version fixes the warning spam caused by scikit-learn parallel warnings and avoids unsafe nested parallelism.

Strict exclusions in this notebook:

- No uncertainty/applicability-domain analysis yet.
- No sodium candidate ranking.
- No CDE matching.
- No criticality filtering.
- No manuscript writing.

In [ ]:
# ============================================================
# Cell 1 — Resource-aware runtime setup, imports, paths
# ============================================================
from __future__ import annotations
import os

# Set thread limits before heavy scientific imports where possible.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import sys, json, math, time, platform, traceback, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# Suppress the repeated scikit-learn parallel warning spam shown in your screenshot.
warnings.filterwarnings(
    "ignore",
    message=r".*sklearn\.utils\.parallel\.delayed.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"sklearn\.utils\.parallel",
)

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

try:
    from threadpoolctl import threadpool_limits
    THREADPOOLCTL_AVAILABLE = True
except Exception:
    THREADPOOLCTL_AVAILABLE = False

from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# -
# Server configuration
# -
SERVER_TOTAL_LOGICAL_CORES = os.cpu_count() or 24

# Your server has 2 x 12 CPU cores. Keep Windows/Jupyter responsive.
# Increase to 20 only if the system remains stable.
N_JOBS_TREE_MODELS = min(18, max(1, SERVER_TOTAL_LOGICAL_CORES - 4))

# First audited run: False. After successful audit, you may set True for extra models.
EXTENDED_MODEL_SET = False

RANDOM_N_SPLITS = 5
GROUP_KFOLD_SPLITS = 5
LEAVE_CHEMSYS_MAX_GROUPS = 75
MIN_TEST_RECORDS_FOR_LEAVEOUT = 5
TREE_N_ESTIMATORS = 300
RANDOM_STATE = 42
CHECKPOINT_EVERY_N_FITS = 25
RESUME_FROM_EXISTING = True

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

NB09_DIR = artifact_namespace("02", REPOSITORY_ROOT)

BASE_DIR = artifact_namespace("04", REPOSITORY_ROOT)
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"
for d in [PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
LOG_ROWS = []

def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "04_event_log.csv", index=False)

def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return "not_installed_or_unknown"

print("Notebook 04 initialized.")
print(f"Using Notebook 02 outputs from: {NB09_DIR}")
print(f"Notebook 04 outputs: {BASE_DIR}")
print(f"Detected logical CPU cores: {SERVER_TOTAL_LOGICAL_CORES}")
print(f"Tree-model n_jobs: {N_JOBS_TREE_MODELS}")
print(f"EXTENDED_MODEL_SET: {EXTENDED_MODEL_SET}")

log_event("init", "INFO", "Notebook 04 initialized.", {
    "nb09_dir": str(NB09_DIR),
    "base_dir": str(BASE_DIR),
    "logical_cores": SERVER_TOTAL_LOGICAL_CORES,
    "n_jobs_tree_models": N_JOBS_TREE_MODELS,
    "extended_model_set": EXTENDED_MODEL_SET,
})
save_event_log()

In [ ]:
# ============================================================
# Cell 2 — Load Notebook 02 outputs
# ============================================================
master_feature_path = NB09_DIR / "processed" / "02_master_feature_table.csv"
metadata_targets_path = NB09_DIR / "processed" / "02_master_metadata_and_targets.csv"
protocol_lookup_path = NB09_DIR / "processed" / "02_protocol_lookup_for_notebook_10.csv"
mask_path = NB09_DIR / "audit" / "02_target_plausibility_masks.csv"
protocol_counts_path = NB09_DIR / "audit" / "02_protocol_feature_counts_by_target.csv"
integrity_path = NB09_DIR / "audit" / "02_protocol_integrity_audit.csv"
final_decision_path = NB09_DIR / "metadata" / "02_final_decision.json"

required_paths = [master_feature_path, metadata_targets_path, protocol_lookup_path, mask_path, protocol_counts_path, integrity_path, final_decision_path]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required Notebook 02 files:\n" + "\n".join(missing_paths))

master_df = pd.read_csv(master_feature_path, low_memory=False)
metadata_targets_df = pd.read_csv(metadata_targets_path, low_memory=False)
protocol_lookup_df = pd.read_csv(protocol_lookup_path)
mask_df = pd.read_csv(mask_path)
protocol_counts_df = pd.read_csv(protocol_counts_path)
integrity_df = pd.read_csv(integrity_path)
with open(final_decision_path, "r", encoding="utf-8") as f:
    nb09_final_decision = json.load(f)

print("Loaded Notebook 02 outputs:")
print(f"  master_df:           {master_df.shape}")
print(f"  metadata_targets_df: {metadata_targets_df.shape}")
print(f"  protocol_lookup_df:  {protocol_lookup_df.shape}")
print(f"  mask_df:             {mask_df.shape}")
print(f"  Notebook 02 decision: {nb09_final_decision.get('final_decision')}")

if nb09_final_decision.get("final_decision") not in {"FULL_GO_TO_NOTEBOOK_10", "CONDITIONAL_GO_TO_NOTEBOOK_04_REVIEW_WARNINGS"}:
    raise RuntimeError(f"Notebook 02 is not cleared for Notebook 04. Decision: {nb09_final_decision.get('final_decision')}")
if (integrity_df["status"] == "FAIL").any():
    raise RuntimeError("Notebook 02 protocol integrity audit contains FAIL rows. Stop.")

display(protocol_counts_df.head(20))
save_event_log()

In [ ]:
# ============================================================
# Cell 3 — Benchmark configuration
# ============================================================
TARGETS = ["average_voltage", "capacity_grav", "energy_grav", "max_delta_volume", "stability_worst"]
PROTOCOLS = ["P0", "P1", "P2", "P3", "P4"]

TARGET_DIRECTION = {
    "average_voltage": "maximize",
    "capacity_grav": "maximize",
    "energy_grav": "maximize",
    "max_delta_volume": "minimize",
    "stability_worst": "minimize",
}
TARGET_UNITS = {
    "average_voltage": "V",
    "capacity_grav": "mAh/g",
    "energy_grav": "Wh/kg",
    "max_delta_volume": "database_volume_change",
    "stability_worst": "eV/atom",
}
TARGET_MASK_COL = {
    "average_voltage": "mask_voltage_0_to_6",
    "capacity_grav": "mask_capacity_grav_positive_le_1000",
    "energy_grav": "mask_energy_grav_nonnegative_le_6000",
    "max_delta_volume": "mask_volume_change_0_to_2",
    "stability_worst": "mask_stability_worst_0_to_2",
}

USE_COMMON_PLAUSIBILITY_MASK = True
COMMON_MASK_COL = "mask_physics_plausible_all_targets"
VALIDATION_SPLITS = ["random_split", "framework_groupkfold", "leave_family_out", "leave_chemical_system_out", "leave_working_ion_out"]

config = {
    "targets": TARGETS,
    "protocols": PROTOCOLS,
    "target_direction": TARGET_DIRECTION,
    "use_common_plausibility_mask": USE_COMMON_PLAUSIBILITY_MASK,
    "common_mask_col": COMMON_MASK_COL,
    "validation_splits": VALIDATION_SPLITS,
    "random_n_splits": RANDOM_N_SPLITS,
    "group_kfold_splits": GROUP_KFOLD_SPLITS,
    "leave_chemsys_max_groups": LEAVE_CHEMSYS_MAX_GROUPS,
    "min_test_records_for_leaveout": MIN_TEST_RECORDS_FOR_LEAVEOUT,
    "n_jobs_tree_models": N_JOBS_TREE_MODELS,
    "extended_model_set": EXTENDED_MODEL_SET,
    "tree_n_estimators": TREE_N_ESTIMATORS,
    "random_state": RANDOM_STATE,
    "outer_loop_parallelism": False,
    "warning_spam_suppression": "enabled",
}
write_json_safe(config, METADATA_DIR / "04_benchmark_config.json")
display(pd.DataFrame([config]))

In [ ]:
# ============================================================
# Cell 4 — Merge masks and create benchmark-ready table
# ============================================================
required_id_cols = ["record_index", "working_ion", "electrode_uid", "framework_uid", "chemical_system_uid", "host_chemsys_no_working_ion", "coarse_family"]
missing_id_cols = [c for c in required_id_cols if c not in master_df.columns]
if missing_id_cols:
    raise ValueError(f"Missing required ID/group columns in master_df: {missing_id_cols}")

required_mask_cols = ["record_index", COMMON_MASK_COL] + list(TARGET_MASK_COL.values())
missing_mask_cols = [c for c in required_mask_cols if c not in mask_df.columns]
if missing_mask_cols:
    raise ValueError(f"Missing required mask columns in mask_df: {missing_mask_cols}")

benchmark_df = master_df.merge(mask_df[required_mask_cols], on="record_index", how="left", validate="one_to_one")
for col in required_mask_cols:
    if col != "record_index":
        benchmark_df[col] = benchmark_df[col].fillna(False).astype(bool)

row_audit = []
for target in TARGETS:
    mask_col = COMMON_MASK_COL if USE_COMMON_PLAUSIBILITY_MASK else TARGET_MASK_COL[target]
    y = pd.to_numeric(benchmark_df[target], errors="coerce")
    valid = benchmark_df[mask_col] & y.notna()
    row_audit.append({
        "target": target,
        "target_unit": TARGET_UNITS.get(target, ""),
        "mask_used": mask_col,
        "n_valid_records": int(valid.sum()),
        "n_total_records": len(benchmark_df),
        "valid_pct": 100.0 * float(valid.sum()) / len(benchmark_df),
        "n_working_ions": int(benchmark_df.loc[valid, "working_ion"].nunique()),
        "n_framework_groups": int(benchmark_df.loc[valid, "framework_uid"].nunique()),
        "n_chemical_systems": int(benchmark_df.loc[valid, "host_chemsys_no_working_ion"].nunique()),
        "n_families": int(benchmark_df.loc[valid, "coarse_family"].nunique()),
    })
row_audit_df = pd.DataFrame(row_audit)
row_audit_df.to_csv(AUDIT_DIR / "04_benchmark_row_audit_by_target.csv", index=False)
display(row_audit_df)

In [ ]:
# ============================================================
# Cell 5 — Model definitions
# ============================================================
def build_models():
    models = {
        "DummyMean": DummyRegressor(strategy="mean"),
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=TREE_N_ESTIMATORS,
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS_TREE_MODELS,
                max_features=1.0,
                min_samples_leaf=1,
                bootstrap=False,
            )),
        ]),
    }
    if EXTENDED_MODEL_SET:
        models.update({
            "RandomForest": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestRegressor(
                    n_estimators=max(200, TREE_N_ESTIMATORS // 2),
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS_TREE_MODELS,
                    max_features=1.0,
                    min_samples_leaf=1,
                    bootstrap=True,
                )),
            ]),
            "HistGradientBoosting": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", HistGradientBoostingRegressor(
                    max_iter=300,
                    learning_rate=0.05,
                    l2_regularization=0.01,
                    random_state=RANDOM_STATE,
                )),
            ]),
        })
    return models

MODELS = build_models()
model_audit_df = pd.DataFrame([{
    "model_name": name,
    "model_repr": repr(model),
    "uses_tree_n_jobs": name in {"ExtraTrees", "RandomForest"},
    "n_jobs_tree_models": N_JOBS_TREE_MODELS if name in {"ExtraTrees", "RandomForest"} else np.nan,
} for name, model in MODELS.items()])
model_audit_df.to_csv(AUDIT_DIR / "04_model_audit.csv", index=False)
display(model_audit_df)
print(f"Models enabled: {list(MODELS.keys())}")

In [ ]:
# ============================================================
# Cell 6 — Feature-list loader and matrix audit
# ============================================================
def load_feature_list(target: str, protocol: str) -> list[str]:
    row = protocol_lookup_df[(protocol_lookup_df["target"] == target) & (protocol_lookup_df["protocol"] == protocol)]
    if row.empty:
        raise ValueError(f"No protocol lookup row for target={target}, protocol={protocol}")
    csv_path = Path(row.iloc[0]["feature_list_csv"])
    if not csv_path.exists():
        candidate = NB09_DIR / "processed" / "protocol_feature_lists" / csv_path.name
        if candidate.exists():
            csv_path = candidate
        else:
            raise FileNotFoundError(f"Feature list not found: {csv_path}")
    flist_df = pd.read_csv(csv_path)
    if "feature" not in flist_df.columns:
        raise ValueError(f"Feature-list CSV missing 'feature' column: {csv_path}")
    return flist_df["feature"].dropna().astype(str).tolist()

FEATURE_CACHE = {}
def get_feature_matrix(target: str, protocol: str):
    key = (target, protocol)
    if key in FEATURE_CACHE:
        return FEATURE_CACHE[key]
    features = load_feature_list(target, protocol)
    available = [f for f in features if f in benchmark_df.columns]
    missing = sorted(set(features) - set(available))
    if missing:
        log_event("feature_matrix", "WARNING", f"Missing features for {target}/{protocol}; dropped.", {"n_missing": len(missing), "missing_sample": missing[:20]})
    if not available:
        raise ValueError(f"No available features for target={target}, protocol={protocol}")
    X = benchmark_df[available].apply(pd.to_numeric, errors="coerce")
    FEATURE_CACHE[key] = (X, available, missing)
    return X, available, missing

feature_matrix_audit_rows = []
for target in TARGETS:
    for protocol in PROTOCOLS:
        X, available, missing = get_feature_matrix(target, protocol)
        feature_matrix_audit_rows.append({
            "target": target,
            "protocol": protocol,
            "n_features_requested": len(available) + len(missing),
            "n_features_available": len(available),
            "n_features_missing": len(missing),
            "n_rows": X.shape[0],
            "n_all_missing_features": int((X.notna().sum(axis=0) == 0).sum()),
        })
feature_matrix_audit_df = pd.DataFrame(feature_matrix_audit_rows)
feature_matrix_audit_df.to_csv(AUDIT_DIR / "04_feature_matrix_audit.csv", index=False)
display(feature_matrix_audit_df)
save_event_log()

In [ ]:
# ============================================================
# Cell 7 — Split-generation functions and split audit
# ============================================================
def _index_array(mask):
    return np.where(np.asarray(mask))[0]

def make_random_splits(valid_indices):
    splitter = ShuffleSplit(n_splits=RANDOM_N_SPLITS, test_size=0.20, random_state=RANDOM_STATE)
    dummy_X = np.zeros((len(valid_indices), 1))
    out = []
    for fold_id, (tr, te) in enumerate(splitter.split(dummy_X)):
        out.append({"split_name": "random_split", "fold_id": fold_id, "heldout_group": f"random_{fold_id}", "train_idx": valid_indices[tr], "test_idx": valid_indices[te], "split_note": "80/20 ShuffleSplit"})
    return out

def make_groupkfold_splits(valid_indices, group_values):
    groups = group_values.iloc[valid_indices].astype(str).fillna("missing_group").values
    n_groups = len(pd.unique(groups))
    if n_groups < 2:
        return []
    n_eff = min(GROUP_KFOLD_SPLITS, n_groups)
    splitter = GroupKFold(n_splits=n_eff)
    dummy_X = np.zeros((len(valid_indices), 1))
    out = []
    for fold_id, (tr, te) in enumerate(splitter.split(dummy_X, groups=groups)):
        heldout_groups = sorted(set(groups[te]))
        out.append({"split_name": "framework_groupkfold", "fold_id": fold_id, "heldout_group": "|".join(heldout_groups[:20]), "train_idx": valid_indices[tr], "test_idx": valid_indices[te], "split_note": f"GroupKFold by framework_uid; n_splits={n_eff}"})
    return out

def make_leave_group_out_splits(valid_indices, group_values, split_name, max_groups=None, min_test_records=MIN_TEST_RECORDS_FOR_LEAVEOUT):
    groups_all = group_values.iloc[valid_indices].astype(str).fillna("missing_group")
    group_counts = groups_all.value_counts()
    eligible = group_counts[group_counts >= min_test_records]
    if max_groups is not None and len(eligible) > max_groups:
        eligible = eligible.sort_values(ascending=False).head(max_groups)
    out = []
    for fold_id, group in enumerate(eligible.index.tolist()):
        is_test = groups_all.values == group
        test_local = np.where(is_test)[0]
        train_local = np.where(~is_test)[0]
        if len(test_local) < min_test_records or len(train_local) < 10:
            continue
        out.append({"split_name": split_name, "fold_id": fold_id, "heldout_group": group, "train_idx": valid_indices[train_local], "test_idx": valid_indices[test_local], "split_note": f"Leave-one-group-out by {split_name}; max_groups={max_groups}"})
    return out

def build_splits_for_target(target):
    mask_col = COMMON_MASK_COL if USE_COMMON_PLAUSIBILITY_MASK else TARGET_MASK_COL[target]
    y = pd.to_numeric(benchmark_df[target], errors="coerce")
    valid_mask = benchmark_df[mask_col].astype(bool) & y.notna()
    valid_indices = _index_array(valid_mask)
    splits = []
    splits.extend(make_random_splits(valid_indices))
    splits.extend(make_groupkfold_splits(valid_indices, benchmark_df["framework_uid"]))
    splits.extend(make_leave_group_out_splits(valid_indices, benchmark_df["coarse_family"], "leave_family_out", max_groups=None))
    splits.extend(make_leave_group_out_splits(valid_indices, benchmark_df["host_chemsys_no_working_ion"], "leave_chemical_system_out", max_groups=LEAVE_CHEMSYS_MAX_GROUPS))
    splits.extend(make_leave_group_out_splits(valid_indices, benchmark_df["working_ion"], "leave_working_ion_out", max_groups=None))
    return splits, valid_indices, mask_col

split_audit_rows = []
for target in TARGETS:
    split_sets, valid_indices, mask_col = build_splits_for_target(target)
    for s in split_sets:
        split_audit_rows.append({"target": target, "split_name": s["split_name"], "fold_id": s["fold_id"], "heldout_group": s["heldout_group"], "n_train": len(s["train_idx"]), "n_test": len(s["test_idx"]), "mask_used": mask_col, "split_note": s["split_note"]})
split_audit_df = pd.DataFrame(split_audit_rows)
split_audit_df.to_csv(AUDIT_DIR / "04_split_composition_audit.csv", index=False)
split_count_summary_df = split_audit_df.groupby(["target", "split_name"], dropna=False).agg(n_folds=("fold_id", "count"), min_train=("n_train", "min"), median_train=("n_train", "median"), min_test=("n_test", "min"), median_test=("n_test", "median")).reset_index()
split_count_summary_df.to_csv(AUDIT_DIR / "04_split_count_summary.csv", index=False)
display(split_count_summary_df)

In [ ]:
# ============================================================
# Cell 8 — Metrics
# ============================================================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_spearman(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 3 or len(np.unique(y_true)) < 2 or len(np.unique(y_pred)) < 2:
        return np.nan
    try:
        val = spearmanr(y_true, y_pred).correlation
        return float(val) if np.isfinite(val) else np.nan
    except Exception:
        return np.nan

def top_k_hit_rate(y_true, y_pred, k, direction):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(y_true)
    if n == 0:
        return np.nan
    k_eff = min(k, n)
    if direction == "minimize":
        true_rank = np.argsort(y_true)[:k_eff]
        pred_rank = np.argsort(y_pred)[:k_eff]
    else:
        true_rank = np.argsort(-y_true)[:k_eff]
        pred_rank = np.argsort(-y_pred)[:k_eff]
    return float(len(set(true_rank) & set(pred_rank)) / k_eff)

def top_fraction_hit_rate(y_true, y_pred, frac, direction):
    k = max(1, int(math.ceil(frac * len(y_true))))
    return top_k_hit_rate(y_true, y_pred, k, direction)

def compute_metrics(y_true, y_pred, target):
    direction = TARGET_DIRECTION.get(target, "maximize")
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan,
        "spearman": safe_spearman(y_true, y_pred),
        "top10_hit_rate": top_k_hit_rate(y_true, y_pred, 10, direction),
        "top20_hit_rate": top_k_hit_rate(y_true, y_pred, 20, direction),
        "top10pct_hit_rate": top_fraction_hit_rate(y_true, y_pred, 0.10, direction),
    }
print("Metric functions ready.")

In [ ]:
# ============================================================
# Cell 9 — Benchmark helpers with checkpoint/resume
# ============================================================
fold_results_path = PROCESSED_DIR / "04_benchmark_fold_results.csv"
predictions_path = PROCESSED_DIR / "04_benchmark_predictions_sample.csv"

def existing_result_keys(path):
    if not path.exists() or not RESUME_FROM_EXISTING:
        return set()
    try:
        df = pd.read_csv(path)
        required = ["target", "protocol", "split_name", "fold_id", "model_name"]
        if not all(c in df.columns for c in required):
            return set()
        return set(zip(df["target"].astype(str), df["protocol"].astype(str), df["split_name"].astype(str), df["fold_id"].astype(int), df["model_name"].astype(str)))
    except Exception:
        return set()

def append_rows_to_csv(rows, path):
    if not rows:
        return
    df = pd.DataFrame(rows)
    df.to_csv(path, mode="a", header=not path.exists(), index=False)

def clean_X_y_for_indices(X, y, indices):
    X_sub = X.iloc[indices].copy()
    y_sub = pd.to_numeric(y.iloc[indices], errors="coerce")
    valid = y_sub.notna().values
    return X_sub.iloc[valid], y_sub.iloc[valid].values.astype(float), indices[valid]

def fit_predict_model(model, X_train, y_train, X_test):
    estimator = clone(model)
    if THREADPOOLCTL_AVAILABLE:
        with threadpool_limits(limits=1):
            estimator.fit(X_train, y_train)
            return estimator.predict(X_test)
    estimator.fit(X_train, y_train)
    return estimator.predict(X_test)

existing_keys = existing_result_keys(fold_results_path)
print(f"Existing completed fold/model results found: {len(existing_keys)}")

In [ ]:
# ============================================================
# Cell 10 — Run validation benchmark
# ============================================================
# Outer loop is sequential. Tree models use n_jobs internally.
# This avoids the sklearn/joblib warning spam and nested parallel instability.
# ============================================================
start_time = time.time()
results_buffer = []
prediction_sample_buffer = []
n_fit_attempts = 0
n_skipped_existing = 0
n_errors = 0

total_jobs_estimate = 0
for target in TARGETS:
    split_sets, _, _ = build_splits_for_target(target)
    total_jobs_estimate += len(PROTOCOLS) * len(split_sets) * len(MODELS)
print(f"Estimated fold/model fits to evaluate: {total_jobs_estimate}")
print("Running benchmark...")

for target in TARGETS:
    y_all = pd.to_numeric(benchmark_df[target], errors="coerce")
    split_sets, valid_indices, mask_col = build_splits_for_target(target)
    print(f"\nTarget: {target} | valid rows: {len(valid_indices)} | splits: {len(split_sets)}")

    for protocol in PROTOCOLS:
        X_all, features, missing_features = get_feature_matrix(target, protocol)
        valid_feature_non_missing = X_all.iloc[valid_indices].notna().sum(axis=0)
        usable_features = valid_feature_non_missing[valid_feature_non_missing > 0].index.tolist()
        if not usable_features:
            log_event("benchmark", "ERROR", f"No usable features for {target}/{protocol}.")
            n_errors += 1
            continue
        X_use = X_all[usable_features]

        for split in split_sets:
            split_name = split["split_name"]
            fold_id = int(split["fold_id"])
            heldout_group = split["heldout_group"]
            train_idx_raw = np.asarray(split["train_idx"], dtype=int)
            test_idx_raw = np.asarray(split["test_idx"], dtype=int)
            X_train, y_train, train_idx = clean_X_y_for_indices(X_use, y_all, train_idx_raw)
            X_test, y_test, test_idx = clean_X_y_for_indices(X_use, y_all, test_idx_raw)
            if len(y_train) < 10 or len(y_test) < 3:
                log_event("benchmark", "WARNING", "Skipping split due to too few records.", {"target": target, "protocol": protocol, "split_name": split_name, "fold_id": fold_id, "n_train": len(y_train), "n_test": len(y_test)})
                continue

            for model_name, model in MODELS.items():
                key = (target, protocol, split_name, fold_id, model_name)
                if key in existing_keys:
                    n_skipped_existing += 1
                    continue
                n_fit_attempts += 1
                try:
                    fit_t0 = time.time()
                    y_pred = fit_predict_model(model, X_train, y_train, X_test)
                    fit_seconds = time.time() - fit_t0
                    metrics = compute_metrics(y_test, y_pred, target)
                    row = {
                        "target": target,
                        "target_unit": TARGET_UNITS.get(target, ""),
                        "target_direction": TARGET_DIRECTION.get(target, "maximize"),
                        "protocol": protocol,
                        "split_name": split_name,
                        "fold_id": fold_id,
                        "heldout_group": heldout_group,
                        "model_name": model_name,
                        "n_train": len(y_train),
                        "n_test": len(y_test),
                        "n_features": len(usable_features),
                        "n_requested_features": len(features),
                        "n_missing_features_from_protocol": len(missing_features),
                        "mask_used": mask_col,
                        "fit_seconds": fit_seconds,
                        "status": "OK",
                        "error": "",
                    }
                    row.update(metrics)
                    results_buffer.append(row)

                    if split_name in {"random_split", "leave_working_ion_out"} and model_name in {"Ridge", "ExtraTrees"}:
                        pred_sample_n = min(200, len(y_test))
                        sample_order = np.arange(pred_sample_n)
                        sample_df = pd.DataFrame({
                            "target": target,
                            "protocol": protocol,
                            "split_name": split_name,
                            "fold_id": fold_id,
                            "heldout_group": heldout_group,
                            "model_name": model_name,
                            "record_index": benchmark_df.iloc[test_idx[sample_order]]["record_index"].values,
                            "working_ion": benchmark_df.iloc[test_idx[sample_order]]["working_ion"].values,
                            "true_value": y_test[sample_order],
                            "predicted_value": y_pred[sample_order],
                        })
                        prediction_sample_buffer.extend(sample_df.to_dict(orient="records"))
                except Exception as exc:
                    n_errors += 1
                    err = f"{type(exc).__name__}: {str(exc)}"
                    log_event("benchmark", "ERROR", "Model fit/predict failed.", {"target": target, "protocol": protocol, "split_name": split_name, "fold_id": fold_id, "model_name": model_name, "error": err, "traceback": traceback.format_exc()[-2000:]})
                    row = {
                        "target": target,
                        "target_unit": TARGET_UNITS.get(target, ""),
                        "target_direction": TARGET_DIRECTION.get(target, "maximize"),
                        "protocol": protocol,
                        "split_name": split_name,
                        "fold_id": fold_id,
                        "heldout_group": heldout_group,
                        "model_name": model_name,
                        "n_train": len(y_train),
                        "n_test": len(y_test),
                        "n_features": len(usable_features),
                        "n_requested_features": len(features),
                        "n_missing_features_from_protocol": len(missing_features),
                        "mask_used": mask_col,
                        "fit_seconds": np.nan,
                        "status": "ERROR",
                        "error": err,
                        "rmse": np.nan,
                        "mae": np.nan,
                        "r2": np.nan,
                        "spearman": np.nan,
                        "top10_hit_rate": np.nan,
                        "top20_hit_rate": np.nan,
                        "top10pct_hit_rate": np.nan,
                    }
                    results_buffer.append(row)

                if len(results_buffer) >= CHECKPOINT_EVERY_N_FITS:
                    append_rows_to_csv(results_buffer, fold_results_path)
                    results_buffer = []
                    if prediction_sample_buffer:
                        append_rows_to_csv(prediction_sample_buffer, predictions_path)
                        prediction_sample_buffer = []
                    save_event_log()
                    elapsed_min = (time.time() - start_time) / 60.0
                    print(f"Checkpoint: attempts={n_fit_attempts}, skipped_existing={n_skipped_existing}, errors={n_errors}, elapsed={elapsed_min:.1f} min")

append_rows_to_csv(results_buffer, fold_results_path)
if prediction_sample_buffer:
    append_rows_to_csv(prediction_sample_buffer, predictions_path)

elapsed_seconds = time.time() - start_time
print("\nBenchmark complete.")
print(f"Fit attempts this run: {n_fit_attempts}")
print(f"Skipped existing: {n_skipped_existing}")
print(f"Errors: {n_errors}")
print(f"Elapsed time: {elapsed_seconds/60:.2f} min")
save_event_log()

In [ ]:
# ============================================================
# Cell 11 — Aggregate fold results and confidence intervals
# ============================================================
if not fold_results_path.exists():
    raise FileNotFoundError(f"Fold results file not found: {fold_results_path}")
fold_results_df = pd.read_csv(fold_results_path)
dedup_keys = ["target", "protocol", "split_name", "fold_id", "model_name"]
fold_results_df = fold_results_df.sort_values(dedup_keys).drop_duplicates(subset=dedup_keys, keep="last")
fold_results_df.to_csv(fold_results_path, index=False)
ok_df = fold_results_df[fold_results_df["status"] == "OK"].copy()

METRIC_COLS = ["rmse", "mae", "r2", "spearman", "top10_hit_rate", "top20_hit_rate", "top10pct_hit_rate"]

def ci95_lower(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    if len(x) <= 1:
        return np.nan
    return float(x.mean() - 1.96 * x.std(ddof=1) / np.sqrt(len(x)))

def ci95_upper(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    if len(x) <= 1:
        return np.nan
    return float(x.mean() + 1.96 * x.std(ddof=1) / np.sqrt(len(x)))

agg_spec = {"fold_id": "count", "n_train": ["min", "median", "max"], "n_test": ["min", "median", "max"], "n_features": "median", "fit_seconds": ["sum", "median"]}
for m in METRIC_COLS:
    agg_spec[m] = ["mean", "std", ci95_lower, ci95_upper]

aggregate_df = ok_df.groupby(["target", "protocol", "split_name", "model_name"], dropna=False).agg(agg_spec)
aggregate_df.columns = ["_".join([str(x) for x in col if str(x) != ""]).strip("_") for col in aggregate_df.columns]
aggregate_df = aggregate_df.reset_index().rename(columns={"fold_id_count": "n_folds", "n_features_median": "median_n_features", "fit_seconds_sum": "total_fit_seconds", "fit_seconds_median": "median_fit_seconds"})
aggregate_path = PROCESSED_DIR / "04_benchmark_aggregate_results.csv"
aggregate_df.to_csv(aggregate_path, index=False)
display(aggregate_df.head(30))
print(f"Saved: {aggregate_path}")

In [ ]:
# ============================================================
# Cell 12 — Best-model table and degradation relative to random split
# ============================================================
best_rows = []
for (target, protocol, split_name), sub in aggregate_df.groupby(["target", "protocol", "split_name"], dropna=False):
    sub2 = sub.dropna(subset=["rmse_mean"]).copy()
    if sub2.empty:
        continue
    best_rows.append(sub2.sort_values("rmse_mean", ascending=True).iloc[0].to_dict())
best_model_df = pd.DataFrame(best_rows)
best_model_path = PROCESSED_DIR / "04_best_model_by_target_protocol_split.csv"
best_model_df.to_csv(best_model_path, index=False)

degradation_rows = []
for (target, protocol, model_name), sub in aggregate_df.groupby(["target", "protocol", "model_name"], dropna=False):
    random_row = sub[sub["split_name"] == "random_split"]
    if random_row.empty:
        continue
    base = random_row.iloc[0]
    for _, row in sub.iterrows():
        degradation_rows.append({
            "target": target,
            "protocol": protocol,
            "model_name": model_name,
            "split_name": row["split_name"],
            "random_rmse_mean": base["rmse_mean"],
            "split_rmse_mean": row["rmse_mean"],
            "rmse_delta_vs_random": row["rmse_mean"] - base["rmse_mean"],
            "rmse_ratio_vs_random": row["rmse_mean"] / base["rmse_mean"] if pd.notna(base["rmse_mean"]) and base["rmse_mean"] != 0 else np.nan,
            "random_mae_mean": base["mae_mean"],
            "split_mae_mean": row["mae_mean"],
            "mae_delta_vs_random": row["mae_mean"] - base["mae_mean"],
            "random_r2_mean": base["r2_mean"],
            "split_r2_mean": row["r2_mean"],
            "r2_delta_vs_random": row["r2_mean"] - base["r2_mean"],
            "n_folds_split": row["n_folds"],
        })
degradation_df = pd.DataFrame(degradation_rows)
degradation_path = PROCESSED_DIR / "04_performance_degradation_relative_to_random.csv"
degradation_df.to_csv(degradation_path, index=False)
display(best_model_df.head(30))
display(degradation_df.head(30))
print(f"Saved: {best_model_path}")
print(f"Saved: {degradation_path}")

In [ ]:
# ============================================================
# Cell 13 — Compact benchmark tables
# ============================================================
compact_cols = ["target", "protocol", "split_name", "model_name", "n_folds", "n_train_median", "n_test_median", "median_n_features", "rmse_mean", "rmse_ci95_lower", "rmse_ci95_upper", "mae_mean", "r2_mean", "spearman_mean", "top10_hit_rate_mean", "top20_hit_rate_mean", "top10pct_hit_rate_mean"]
compact_cols = [c for c in compact_cols if c in best_model_df.columns]
compact_df = best_model_df[compact_cols].copy()
compact_df.to_csv(PROCESSED_DIR / "04_compact_best_model_benchmark_table.csv", index=False)

model_comparison_cols = ["target", "protocol", "split_name", "model_name", "n_folds", "n_train_median", "n_test_median", "median_n_features", "rmse_mean", "mae_mean", "r2_mean", "spearman_mean", "top10_hit_rate_mean", "top20_hit_rate_mean", "top10pct_hit_rate_mean", "total_fit_seconds"]
model_comparison_cols = [c for c in model_comparison_cols if c in aggregate_df.columns]
model_comparison_df = aggregate_df[model_comparison_cols].copy()
model_comparison_df.to_csv(PROCESSED_DIR / "04_model_comparison_table.csv", index=False)
display(compact_df.head(50))
print("Saved compact benchmark tables.")

In [ ]:
# ============================================================
# Cell 14 — Error and execution coverage audit
# ============================================================
error_df = fold_results_df[fold_results_df["status"] != "OK"].copy()
if not error_df.empty:
    error_summary_df = error_df.groupby(["target", "protocol", "split_name", "model_name", "error"], dropna=False).size().reset_index(name="n_errors").sort_values("n_errors", ascending=False)
else:
    error_summary_df = pd.DataFrame(columns=["target", "protocol", "split_name", "model_name", "error", "n_errors"])
error_df.to_csv(AUDIT_DIR / "04_fold_error_rows.csv", index=False)
error_summary_df.to_csv(AUDIT_DIR / "04_error_summary.csv", index=False)

coverage_df = fold_results_df.groupby(["target", "protocol", "split_name", "model_name"], dropna=False).agg(n_rows=("status", "count"), n_ok=("status", lambda x: int((x == "OK").sum())), n_error=("status", lambda x: int((x != "OK").sum()))).reset_index()
coverage_df.to_csv(AUDIT_DIR / "04_execution_coverage_audit.csv", index=False)
display(error_summary_df.head(20))
display(coverage_df.head(30))

In [ ]:
# ============================================================
# Cell 15 — Final decision and output manifest
# ============================================================
def list_output_files(base_dir: Path):
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append({"relative_path": str(path.relative_to(base_dir)), "size_bytes": path.stat().st_size, "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")})
    return pd.DataFrame(rows)

required_outputs = [
    PROCESSED_DIR / "04_benchmark_fold_results.csv",
    PROCESSED_DIR / "04_benchmark_aggregate_results.csv",
    PROCESSED_DIR / "04_best_model_by_target_protocol_split.csv",
    PROCESSED_DIR / "04_performance_degradation_relative_to_random.csv",
    PROCESSED_DIR / "04_compact_best_model_benchmark_table.csv",
    PROCESSED_DIR / "04_model_comparison_table.csv",
    AUDIT_DIR / "04_split_composition_audit.csv",
    AUDIT_DIR / "04_split_count_summary.csv",
    AUDIT_DIR / "04_feature_matrix_audit.csv",
    AUDIT_DIR / "04_execution_coverage_audit.csv",
    AUDIT_DIR / "04_error_summary.csv",
    METADATA_DIR / "04_benchmark_config.json",
]
missing_outputs = [str(p) for p in required_outputs if not p.exists()]
n_error_rows = int((fold_results_df["status"] != "OK").sum()) if "fold_results_df" in globals() else -1
n_ok_rows = int((fold_results_df["status"] == "OK").sum()) if "fold_results_df" in globals() else 0
expected_split_names = set(VALIDATION_SPLITS)
observed_split_names = set(fold_results_df.loc[fold_results_df["status"] == "OK", "split_name"].unique()) if "fold_results_df" in globals() else set()
missing_split_names = sorted(expected_split_names - observed_split_names)

if missing_outputs:
    FINAL_DECISION_10 = "NO_GO_MISSING_OUTPUTS"
elif n_ok_rows == 0:
    FINAL_DECISION_10 = "NO_GO_NO_SUCCESSFUL_FITS"
elif missing_split_names:
    FINAL_DECISION_10 = "CONDITIONAL_GO_REVIEW_MISSING_SPLITS"
elif n_error_rows > 0:
    FINAL_DECISION_10 = "CONDITIONAL_GO_REVIEW_ERRORS"
else:
    FINAL_DECISION_10 = "FULL_GO_TO_NOTEBOOK_11"

final_decision = {
    "final_decision": FINAL_DECISION_10,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "n_ok_rows": n_ok_rows,
    "n_error_rows": n_error_rows,
    "missing_outputs": missing_outputs,
    "observed_split_names": sorted(observed_split_names),
    "missing_split_names": missing_split_names,
    "n_jobs_tree_models": N_JOBS_TREE_MODELS,
    "extended_model_set": EXTENDED_MODEL_SET,
    "tree_n_estimators": TREE_N_ESTIMATORS,
    "outer_loop_parallelism": False,
    "warning_spam_suppression": "enabled for sklearn.utils.parallel.delayed UserWarning",
    "no_uncertainty_ad_analysis_performed": True,
    "no_candidate_ranking_performed": True,
    "no_cde_matching_performed": True,
    "no_criticality_filtering_performed": True,
    "no_manuscript_writing_performed": True,
}
write_json_safe(final_decision, METADATA_DIR / "04_final_decision.json")
output_manifest_df = list_output_files(BASE_DIR)
output_manifest_df.to_csv(METADATA_DIR / "04_output_file_manifest.csv", index=False)
display(output_manifest_df)
print("\n" + "=" * 80)
print(f"Notebook 04 FINAL DECISION: {FINAL_DECISION_10}")
print("=" * 80)
print("\nKey outputs:")
for p in required_outputs:
    print(f" - {p}  {'[OK]' if p.exists() else '[MISSING]'}")
save_event_log()
print("\nNotebook 10 complete.")